In [1]:
import os
import boto3
from botocore import UNSIGNED
from botocore.config import Config

def descargar_nsrdb_puerto_rico(directorio_destino='Data/datos_nsrdb'):
    """
    Descarga archivos HDF5 del dataset NSRDB para Puerto Rico desde AWS S3.
    Utiliza acceso anónimo (UNSIGNED) ya que el bucket de OpenEI es público.
    """
    bucket_name = 'nrel-pds-nsrdb'
    prefix = 'puerto_rico/'
    
    # Crear el directorio de destino si no existe
    os.makedirs(directorio_destino, exist_ok=True)
    
    # Configurar el cliente S3 para solicitudes anónimas
    s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    
    # Inicializar el paginador para manejar más de 1000 objetos si es necesario
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    
    print(f"Conectando al bucket '{bucket_name}' y buscando en '{prefix}'...\n")
    
    archivos_descargados = 0
    
    for page in pages:
        if 'Contents' not in page:
            continue
            
        for obj in page['Contents']:
            key = obj['Key']
            
            # Filtrar para descargar solo los archivos de datos (HDF5)
            if key.endswith('.h5'):
                filename = os.path.basename(key)
                local_path = os.path.join(directorio_destino, filename)
                
                # Evitar descargar archivos que ya existen localmente
                if os.path.exists(local_path):
                    print(f"Saltando {filename} (Ya existe localmente)")
                    continue
                
                print(f"Descargando: {filename} ({obj['Size'] / (1024*1024):.2f} MB)")
                
                try:
                    s3_client.download_file(bucket_name, key, local_path)
                    archivos_descargados += 1
                except Exception as e:
                    print(f"Error al descargar {filename}: {e}")
                    
    print(f"\nProceso completado. Se descargaron {archivos_descargados} archivos nuevos.")



In [2]:
if __name__ == "__main__":
    # Ejecutar el script
    descargar_nsrdb_puerto_rico()

Conectando al bucket 'nrel-pds-nsrdb' y buscando en 'puerto_rico/'...

Descargando: nsrdb_puerto_rico_1998.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_1999.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2000.h5 (5865.56 MB)
Descargando: nsrdb_puerto_rico_2001.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2002.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2003.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2004.h5 (5865.56 MB)
Descargando: nsrdb_puerto_rico_2005.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2006.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2007.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2008.h5 (5865.56 MB)
Descargando: nsrdb_puerto_rico_2009.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2010.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2011.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2012.h5 (5865.56 MB)
Descargando: nsrdb_puerto_rico_2013.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2014.h5 (5865.55 MB)
Descargando: nsrdb_puerto_rico_2015.h5 (5865.

### Obtener datos de cloud_type and relative_humidity

In [ ]:
#%pip install python-dotenv requests pyarrow

In [ ]:
import os
import time
import h5py
import requests
import pandas as pd
import io
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()
API_KEY = os.getenv('API_KEY')

# Configuración del usuario (correo)
EMAIL_USUARIO = "a2173010088@alumnos.uat.edu.mx" 

# ==========================================
# 1. Extracción de Coordenadas
# ==========================================
def obtener_coordenadas(h5_path):
    """Extrae la latitud y longitud de los 2480 nodos desde el archivo original."""
    print("Extrayendo malla espacial del archivo HDF5...")
    with h5py.File(h5_path, 'r') as f:
        meta_df = pd.DataFrame(f['meta'][:])
        
    # Decodificar bytes si es necesario (depende de cómo esté codificado el HDF5)
    if isinstance(meta_df['latitude'].iloc[0], bytes):
        meta_df['latitude'] = meta_df['latitude'].apply(lambda x: float(x.decode('utf-8')))
        meta_df['longitude'] = meta_df['longitude'].apply(lambda x: float(x.decode('utf-8')))
        
    return meta_df[['latitude', 'longitude']]

# ==========================================
# 2. Motor de Descarga API
# ==========================================
def descargar_datos_nodo(nodo_id, lat, lon, anios, atributos, dir_salida):
    """Realiza la petición a la API del NLR y guarda el resultado en Parquet."""
    url_base = "https://developer.nlr.gov/api/solar/nsrdb_psm3_download.csv"
    
    # Parámetros de la URL
    parametros = {
        'api_key': API_KEY,
        'email': EMAIL_USUARIO,
        'lat': lat,
        'lon': lon,
        'names': anios,
        'attributes': atributos,
        'interval': '60', # Promedios horarios
        'utc': 'true'     # Mantener la misma zona horaria que los archivos h5
    }
    
    try:
        response = requests.get(url_base, params=parametros, timeout=30)
        
        # Manejo de Límite Diario (Error 429)
        if response.status_code == 429:
            print("\n❌ [ERROR 429] Límite diario de 2,000 peticiones alcanzado.")
            return 'LIMITE_ALCANZADO'
            
        # Manejo de otros errores
        response.raise_for_status()
        
        # La API de PSM3 típicamente devuelve 2 filas de metadatos antes de los encabezados
        # Saltamos esas 2 filas para que pandas lea directamente los datos
        df_nodo = pd.read_csv(io.StringIO(response.text), skiprows=2)
        
        # Añadir el identificador del nodo
        df_nodo.insert(0, 'nodo_id', nodo_id)
        
        # Guardar como Parquet
        archivo_salida = os.path.join(dir_salida, f"nodo_{nodo_id}.parquet")
        df_nodo.to_parquet(archivo_salida, engine='pyarrow', index=False)
        
        return 'EXITO'
        
    except requests.exceptions.RequestException as e:
        print(f"  -> Error de red en el nodo {nodo_id}: {e}")
        return 'ERROR'

# ==========================================
# 3. Orquestador Principal
# ==========================================
def iniciar_pipeline():
    # Rutas
    archivo_h5_referencia = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5'
    directorio_salida = 'Data/API_Historico'
    archivo_checkpoint = 'Data/nodos_completados.log'
    
    # Parámetros de la consulta
    anios_objetivo = '2013,2014,2015,2016,2017'
    atributos_objetivo = 'cloud_type,relative_humidity'
    
    os.makedirs(directorio_salida, exist_ok=True)
    
    # 1. Obtener coordenadas
    df_coordenadas = obtener_coordenadas(archivo_h5_referencia)
    
    # 2. Leer Checkpoints para saber dónde reanudar
    nodos_completados = set()
    if os.path.exists(archivo_checkpoint):
        with open(archivo_checkpoint, 'r') as f:
            nodos_completados = set(int(line.strip()) for line in f.readlines())
            
    nodos_pendientes = [n for n in df_coordenadas.index if n not in nodos_completados]
    
    print(f"\n🚀 Iniciando extracción API NSRDB")
    print(f"Total de nodos: {len(df_coordenadas)}")
    print(f"Ya descargados: {len(nodos_completados)}")
    print(f"Por descargar hoy: {len(nodos_pendientes)}\n")
    print("-" * 50)
    
    # 3. Bucle de Descarga
    for nodo_id in nodos_pendientes:
        lat = df_coordenadas.loc[nodo_id, 'latitude']
        lon = df_coordenadas.loc[nodo_id, 'longitude']
        
        print(f"Solicitando Nodo {nodo_id:04d} | Lat: {lat:.4f}, Lon: {lon:.4f}...", end=" ")
        
        estado = descargar_datos_nodo(nodo_id, lat, lon, anios_objetivo, atributos_objetivo, directorio_salida)
        
        if estado == 'LIMITE_ALCANZADO':
            print("Deteniendo script por hoy. Tu progreso está guardado.")
            break
            
        elif estado == 'EXITO':
            print("✅ Descargado")
            # Guardar checkpoint
            with open(archivo_checkpoint, 'a') as f:
                f.write(f"{nodo_id}\n")
                
        # Pausa vitalicia para evitar bloqueos por DDoS (Rate Limiting)
        time.sleep(1.5)

if __name__ == "__main__":
    if not API_KEY:
        print("❌ Error: No se encontró la API_KEY en el archivo .env")
    else:
        iniciar_pipeline()

In [11]:
import os
import io
import h5py
import requests
import pandas as pd
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()
API_KEY = os.getenv('API_KEY')

# ==========================================
# Inyecta aquí tu correo institucional (@alumnos.uat.edu.mx)
# ==========================================
EMAIL_USUARIO = "a2173010088@alumnos.uat.edu.mx" 

def prueba_api_v4_nodo_cero():
    archivo_h5 = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5'
    
    print("Obteniendo coordenadas del Nodo 0...")
    try:
        with h5py.File(archivo_h5, 'r') as f:
            lat = f['meta'][0]['latitude']
            lon = f['meta'][0]['longitude']
            
            if isinstance(lat, bytes):
                lat = float(lat.decode('utf-8'))
                lon = float(lon.decode('utf-8'))
    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo {archivo_h5}.")
        return

    print(f"Coordenadas -> Latitud: {lat:.4f}, Longitud: {lon:.4f}")

    # 1. El endpoint oficial activo
    url_base = "https://developer.nlr.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download.csv"
    
    # 2. String WKT requerido por la API v4 (Longitud separada por un espacio de la Latitud)
    wkt_punto = f"POINT({lon} {lat})"
    
    # 3. Parámetros ajustados: Se elimina 'lat' y 'lon', se usa exclusivamente 'wkt'
    parametros = {
        'api_key': API_KEY,
        'email': EMAIL_USUARIO,
        'wkt': wkt_punto,
        'names': '2017', 
        'attributes': 'cloud_type,relative_humidity',
        'interval': '60',       
        'utc': 'true',
        'leap_day': 'false'
    }

    print("\nRealizando petición GET a la API PSM v4 con formato WKT...")
    
    # Ejecutamos la petición
    response = requests.get(url_base, params=parametros)
    
    # Manejo de errores
    if response.status_code != 200:
        print(f"❌ Error HTTP {response.status_code}")
        print("Mensaje del servidor:", response.text)
        return

    print("✅ Petición exitosa. Procesando datos...")
    
    # La API devuelve 2 líneas de metadatos antes de la tabla, las saltamos
    df = pd.read_csv(io.StringIO(response.text), skiprows=2)
    
    # Limpiamos los nombres de las columnas por seguridad
    df.columns = df.columns.str.strip()
    
    # Inyectamos el ID del nodo
    df.insert(0, 'nodo_id', 0)

    # Guardar archivo de prueba
    os.makedirs('Data/API_Historico', exist_ok=True)
    ruta_salida = 'Data/API_Historico/prueba_nodo_0_2017.parquet'
    df.to_parquet(ruta_salida, engine='pyarrow', index=False)
    
    print(f"💾 Archivo guardado en: {ruta_salida}\n")

    # Mostrar resultados
    print("="*60)
    print("VISTA PREVIA DE LOS DATOS DESCARGADOS")
    print("="*60)
    print(df.head())
    print(f"\nTotal de filas: {len(df):,} (Debe ser exactamente 8,760 para un año horario)")

if __name__ == "__main__":
    if not API_KEY:
        print("❌ Error: Verifica tu archivo .env, no se detectó API_KEY.")
    else:
        prueba_api_v4_nodo_cero()

Obteniendo coordenadas del Nodo 0...
Coordenadas -> Latitud: 18.1200, Longitud: -67.9300

Realizando petición GET a la API PSM v4 con formato WKT...
✅ Petición exitosa. Procesando datos...
💾 Archivo guardado en: Data/API_Historico/prueba_nodo_0_2017.parquet

VISTA PREVIA DE LOS DATOS DESCARGADOS
   nodo_id  Year  Month  Day  Hour  Minute  Cloud Type  Relative Humidity
0        0  2017      1    1     0      30           1              82.75
1        0  2017      1    1     1      30           1              84.37
2        0  2017      1    1     2      30           1              84.99
3        0  2017      1    1     3      30           3              84.98
4        0  2017      1    1     4      30           3              84.30

Total de filas: 8,760 (Debe ser exactamente 8,760 para un año horario)


### Descarga de datos API - Generación PSM v4

In [1]:
import os
import io
import h5py
import requests
import pandas as pd
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()
API_KEY = os.getenv('API_KEY')
EMAIL_USUARIO = "a2173010088@alumnos.uat.edu.mx" # <-- Tu correo real

def prueba_api_v4_nodo_cero_completo():
    archivo_h5 = 'Data/datos_nsrdb/nsrdb_puerto_rico_2017.h5'
    
    print("Obteniendo coordenadas del Nodo 0...")
    try:
        with h5py.File(archivo_h5, 'r') as f:
            lat = f['meta'][0]['latitude']
            lon = f['meta'][0]['longitude']
            
            if isinstance(lat, bytes):
                lat = float(lat.decode('utf-8'))
                lon = float(lon.decode('utf-8'))
    except FileNotFoundError:
        print(f"❌ Error: No se encontró el archivo {archivo_h5}.")
        return

    # 1. Endpoint oficial activo (PSM v4.0.0)
    url_base = "https://developer.nlr.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download.csv"
    wkt_punto = f"POINT({lon} {lat})"
    
    # 2. Parámetros ajustados: SIN la llave 'attributes' para pedir todo el dataset
    parametros = {
        'api_key': API_KEY,
        'email': EMAIL_USUARIO,
        'wkt': wkt_punto,
        'names': '2017', 
        'interval': '60',       
        'utc': 'true',
        'leap_day': 'false'
    }

    print(f"\nRealizando petición GET a la API PSM v4 para TODAS las variables...")
    print(f"Coordenadas: Lat {lat:.4f}, Lon {lon:.4f}")
    
    response = requests.get(url_base, params=parametros)
    
    if response.status_code != 200:
        print(f"❌ Error HTTP {response.status_code}")
        print("Mensaje del servidor:", response.text)
        return

    print("✅ Petición exitosa. Procesando datos...")
    
    # Procesar CSV
    df = pd.read_csv(io.StringIO(response.text), skiprows=2)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.insert(0, 'nodo_id', 0)

    # Guardar archivo de prueba
    os.makedirs('Data/API_Historico', exist_ok=True)
    ruta_salida = 'Data/API_Historico/prueba_nodo_0_2017_completo.parquet'
    df.to_parquet(ruta_salida, engine='pyarrow', index=False)
    
    print(f"💾 Archivo guardado en: {ruta_salida}\n")

    # Mostrar resultados
    print("="*80)
    print("VISTA PREVIA: TODAS LAS VARIABLES (Primeras 5 filas)")
    print("="*80)
    
    # Configuramos Pandas para que no oculte columnas en la consola
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    print(df.head())
    print("\n" + "="*80)
    print(f"Estructura final: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    print(f"Lista de columnas descargadas:\n{list(df.columns)}")

if __name__ == "__main__":
    if not API_KEY:
        print("❌ Error: Verifica tu archivo .env, no se detectó API_KEY.")
    else:
        prueba_api_v4_nodo_cero_completo()

Obteniendo coordenadas del Nodo 0...

Realizando petición GET a la API PSM v4 para TODAS las variables...
Coordenadas: Lat 18.1200, Lon -67.9300
✅ Petición exitosa. Procesando datos...
💾 Archivo guardado en: Data/API_Historico/prueba_nodo_0_2017_completo.parquet

VISTA PREVIA: TODAS LAS VARIABLES (Primeras 5 filas)
   nodo_id  year  month  day  hour  minute  temperature  alpha  aerosol_optical_depth  asymmetry  clearsky_dhi  clearsky_dni  clearsky_ghi  cloud_fill_flag  cloud_type  dew_point  dhi  dni  fill_flag  ghi  ozone  relative_humidity  solar_zenith_angle   ssa  surface_albedo  pressure  precipitable_water  wind_direction  wind_speed  global_horizontal_uv_irradiance_(280-400nm)  global_horizontal_uv_irradiance_(295-385nm)
0        0  2017      1    1     0      30         27.0   0.35                  0.043       0.69             0             0             0                0           1       23.8    0    0          0    0  0.239              82.75              122.80  0.98      

In [ ]:
!python Utils/puerto_rico_v4_2017/consolidar_limpiar_v4.py

In [ ]:
!python Utils/puerto_rico_v4_2017/filtro_diurno_v4.py

In [ ]:
import pandas as pd

# Ruta al dataset definitivo con filtros astronómicos aplicados
ruta_dataset = 'Data/Puerto_Rico_v4_2017/Finales/filtrado/dataset_v4_filtrado_diurno_2017.parquet'

try:
    # Cargar el dataframe optimizado en formato Parquet
    df = pd.read_parquet(ruta_dataset, engine='pyarrow')
    
    # Filtrar las primeras 24 filas diurnas correspondientes al Nodo 0
    df_ejemplo = df[df['nodo_id'] == 0].head(24)
    
    print("=" * 100)
    print("MUESTRA DE DATOS CURADOS: PRIMERAS FILAS DIURNAS (NODO 0)")
    print("=" * 100)
    
    # Configurar Pandas para mostrar el espectro completo de variables
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    # Imprimir la matriz omitiendo el índice secuencial de Pandas
    print(df_ejemplo.to_string(index=False))
    print("=" * 100)

except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo en la ruta: {ruta_dataset}")
except Exception as e:
    print(f"❌ Ocurrió un fallo inesperado al leer el dataset: {e}")